# OpenForensics Wild + generated masks — compare

This notebook does not train and does not load images.

Attach the latest output of each `OW_01`–`OW_04` notebook that has finished a test cell.
Each output contains `experiment_results.csv`.

Classification numbers are Wild folder labels.
Dice and mIoU, when present, are agreement with the generated masks.


In [ ]:
import json
from pathlib import Path

import pandas as pd

EXPECTED = [
    "baseline_effb7_ow",
    "multistream_no_seg_ow",
    "seg_no_multistream_ow",
    "full_veritas_ow",
]
roots = [p for p in (Path("/kaggle/input"), Path("/kaggle/working"), Path.cwd(), Path.cwd() / "kaggle_working") if p.exists()]
frames = []
for root in roots:
    for path in root.rglob("experiment_results.csv"):
        try:
            df = pd.read_csv(path)
        except Exception as e:
            print("skip", path, e)
            continue
        if "experiment" not in df.columns:
            continue
        hit = df[df.experiment.astype(str).isin(EXPECTED)].copy()
        if hit.empty:
            continue
        hit["source"] = str(path)
        frames.append(hit)
        print("read", path, hit.experiment.tolist())
if not frames:
    raise FileNotFoundError("No OW experiment_results.csv found. Attach the OW_01–OW_04 outputs after their test cells have run.")
all_df = pd.concat(frames, ignore_index=True)
all_df = all_df.drop_duplicates("experiment", keep="last")
cols = [c for c in ["experiment", "epochs_completed", "n", "accuracy", "f1", "precision", "recall", "tp", "tn", "fp", "fn", "dice", "miou"] if c in all_df.columns]
print(all_df[cols].sort_values("experiment").to_string(index=False))
missing = [e for e in EXPECTED if e not in set(all_df.experiment.astype(str))]
if missing:
    print("still missing:", missing)
out = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd() / "kaggle_working"
out.mkdir(parents=True, exist_ok=True)
all_df.to_csv(out / "ow_compare.csv", index=False)
print("wrote", out / "ow_compare.csv")
print("Dice and mIoU compare each segmentation model with the generated mask. They are not a hand-labeled localization score.")
